# ⚡ Adaptive Flow Engine - Colab GPU Stem Processor

Notebook này sử dụng GPU miễn phí của Google Colab (NVIDIA Tesla T4 / V100) để tách bài hát thành 4 stems độc lập chất lượng cao (**vocals, other, bass, drums**) định dạng MP3 320kbps + phân tích BPM và phổ năng lượng âm thanh.

---
### 🎯 LỰA CHỌN PHƯƠNG THỨC HOẠT ĐỘNG:
1. **Chế độ 1 (Khuyên dùng khi có Google Drive Desktop):** Tự động đồng bộ ngầm 2 chiều giữa App và Drive.
2. **Chế độ 2 (Không cần Google Drive Desktop - Web Upload & ZIP):** Bạn chọn tải các file nhạc từ máy lên, Colab tách xong sẽ tự động nén thành `Stems_Completed.zip` và tải về máy tính để bạn nạp vào App.

In [ ]:
# 1. Cài đặt các thư viện AI & Xử lý Âm thanh
print("⏳ Đang cài đặt Demucs AI, Librosa, Pydub...")
!pip install -q demucs librosa soundfile pydub

import torch
print("\n--- KIỂM TRA PHẦN CỨNG ---")
if torch.cuda.is_available():
    print(f"✅ Đã kích hoạt GPU: {torch.cuda.get_device_name(0)}")
    print(f"⚡ Bộ nhớ VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("⚠️ CẢNH BÁO: Bạn chưa bật GPU! Vào menu Runtime -> Change runtime type -> Chọn T4 GPU để tăng tốc độ gấp 15 lần!")

## 📂 CHẾ ĐỘ 1: TỰ ĐỒNG BỘ QUA GOOGLE DRIVE
*(Nếu máy tính của bạn đã cài đặt Google Drive for Desktop)*

Chỉ cần bấm Chạy (Play) 2 cell bên dưới, bài hát trong `AdaptiveMusic_Sync/01_Input_Audio` sẽ tự động tách và chuyển về `03_Output_Stems`.

In [ ]:
# Chế độ 1 - Bước 1: Kết nối Google Drive và chuẩn bị thư mục
import os
from google.colab import drive

print("Đang kết nối Google Drive...")
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/AdaptiveMusic_Sync"
INPUT_DIR = os.path.join(BASE_DIR, "01_Input_Audio")
PROCESSED_DIR = os.path.join(BASE_DIR, "02_Processed_Audio")
OUTPUT_DIR = os.path.join(BASE_DIR, "03_Output_Stems")

for d in [INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Thư mục Google Drive đã sẵn sàng tại: {BASE_DIR}")

In [ ]:
# Chế độ 1 - Bước 2: BATCH PROCESSOR tự động quét và tách nhạc trên Drive
import glob
import json
import shutil
import subprocess
import urllib.request
import librosa
from pydub import AudioSegment

input_files = glob.glob(os.path.join(INPUT_DIR, "*.mp3")) + glob.glob(os.path.join(INPUT_DIR, "*.wav")) + glob.glob(os.path.join(INPUT_DIR, "*.m4a")) + glob.glob(os.path.join(INPUT_DIR, "*.flac"))
total = len(input_files)

print(f"🎵 Tìm thấy {total} bài hát đang chờ trong 01_Input_Audio/\n")

for idx, file_path in enumerate(input_files, 1):
    filename = os.path.basename(file_path)
    video_id = os.path.splitext(filename)[0]
    print(f"\n[{idx}/{total}] 🚀 Bắt đầu xử lý: {filename} (ID: {video_id})")

    target_stem_dir = os.path.join(OUTPUT_DIR, video_id)
    os.makedirs(target_stem_dir, exist_ok=True)

    # Cache check
    has_all_stems = all(os.path.exists(os.path.join(target_stem_dir, f"{s}.mp3")) for s in ["vocals", "other", "bass", "drums"])
    if has_all_stems:
        print(f"  ⚡ Cache Hit: Bài này đã có đủ 4 stems. Di chuyển sang 02_Processed_Audio.")
        shutil.move(file_path, os.path.join(PROCESSED_DIR, filename))
        continue

    temp_demucs_dir = f"/content/temp_demucs_{video_id}"
    os.makedirs(temp_demucs_dir, exist_ok=True)

    try:
        cmd = [
            "python3", "-m", "demucs.separate",
            "-n", "htdemucs",
            "-d", "cuda" if torch.cuda.is_available() else "cpu",
            "-o", temp_demucs_dir,
            file_path
        ]
        subprocess.run(cmd, check=True)

        stems_folder = os.path.join(temp_demucs_dir, "htdemucs", os.path.splitext(filename)[0])
        if not os.path.exists(stems_folder):
            subdirs = [os.path.join(temp_demucs_dir, "htdemucs", d) for d in os.listdir(os.path.join(temp_demucs_dir, "htdemucs")) if os.path.isdir(os.path.join(temp_demucs_dir, "htdemucs", d))]
            if subdirs:
                stems_folder = subdirs[0]

        # Đo BPM & RMS
        print("  📊 Đang phân tích BPM và phổ âm...")
        y, sr = librosa.load(file_path, sr=22050, duration=90)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        bpm = float(tempo[0]) if hasattr(tempo, "__iter__") else float(tempo)
        rms_energy = float(librosa.feature.rms(y=y).mean())

        print("  🎛️ Đang xuất 4 stems MP3 320kbps...")
        other_seg = AudioSegment.from_wav(os.path.join(stems_folder, "other.wav"))
        drums_seg = AudioSegment.from_wav(os.path.join(stems_folder, "drums.wav"))
        bass_seg = AudioSegment.from_wav(os.path.join(stems_folder, "bass.wav"))
        vocals_seg = AudioSegment.from_wav(os.path.join(stems_folder, "vocals.wav"))

        min_len = min(len(other_seg), len(drums_seg), len(bass_seg), len(vocals_seg))
        vocals_seg[:min_len].export(os.path.join(target_stem_dir, "vocals.mp3"), format="mp3", bitrate="320k")
        other_seg[:min_len].export(os.path.join(target_stem_dir, "other.mp3"), format="mp3", bitrate="320k")
        bass_seg[:min_len].export(os.path.join(target_stem_dir, "bass.mp3"), format="mp3", bitrate="320k")
        drums_seg[:min_len].export(os.path.join(target_stem_dir, "drums.mp3"), format="mp3", bitrate="320k")

        track_title = f"Track {video_id}"
        try:
            oembed_url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
            req = urllib.request.Request(oembed_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=3) as r:
                d = json.loads(r.read().decode())
                if d.get("title"):
                    track_title = d["title"]
        except Exception:
            pass

        meta = {
            "video_id": video_id,
            "title": track_title,
            "duration": int(min_len / 1000),
            "bpm": round(bpm, 1),
            "rms_energy": round(rms_energy, 4),
            "bitrate": "320kbps",
            "processed_by": "Google Colab HTDemucs GPU (Drive Sync)",
            "status": "READY"
        }
        with open(os.path.join(target_stem_dir, "metadata.json"), "w", encoding="utf-8") as mf:
            json.dump(meta, mf, indent=2, ensure_ascii=False)

        shutil.move(file_path, os.path.join(PROCESSED_DIR, filename))
        print(f"  ✅ Hoàn tất bài: {track_title} (BPM: {round(bpm, 1)})")
    except Exception as e:
        print(f"  ❌ Lỗi xử lý {filename}: {e}")
    finally:
        shutil.rmtree(temp_demucs_dir, ignore_errors=True)

print("\n🎉 XONG TOÀN BỘ DANH SÁCH DRIVE! Google Drive Desktop sẽ tự động đồng bộ kết quả về App.")

## 🌐 CHẾ ĐỘ 2: UPLOAD TRỰC TIẾP & TẢI FILE ZIP
*(Dành cho máy KHÔNG cài Google Drive for Desktop)*

1. Bấm nút **Play** ở cell bên dưới.
2. Bấm **Choose Files** và chọn các bài hát từ máy của bạn (hoặc từ thư mục `01_Input_Audio` mà App vừa mở cho bạn).
3. Colab sẽ tự động tách 4 stems (320kbps), tính BPM, đóng gói vào file `Stems_Completed.zip` và tự động tải về máy.
4. Sau khi tải xong, bạn mở ứng dụng Adaptive Music, bấm `[📥 Nạp File ZIP Stems]` và kéo file zip vào là xong!

In [ ]:
# Chế độ 2: Tải file lên trực tiếp qua Web -> Tách 4 stems GPU -> Xuất ZIP tải về
import os
import glob
import json
import shutil
import zipfile
import subprocess
import urllib.request
import librosa
from pydub import AudioSegment
from google.colab import files

print("👉 BƯỚC 1: Hãy chọn các file nhạc (.mp3, .wav, .m4a, .flac) cần tách từ máy tính...")
uploaded = files.upload()

if not uploaded:
    print("⚠️ Bạn chưa chọn file nào. Hãy chạy lại cell này khi muốn tải nhạc lên.")
else:
    WORK_DIR = "/content/web_stems_workspace"
    INPUT_DIR = os.path.join(WORK_DIR, "inputs")
    OUTPUT_DIR = os.path.join(WORK_DIR, "Stems_Completed")
    os.makedirs(INPUT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for fname, data in uploaded.items():
        target_path = os.path.join(INPUT_DIR, fname)
        with open(target_path, "wb") as f:
            f.write(data)

    input_files = glob.glob(os.path.join(INPUT_DIR, "*.*"))
    total = len(input_files)
    print(f"\n🚀 Bắt đầu tách {total} bài hát bằng GPU Tesla T4...")

    for idx, file_path in enumerate(input_files, 1):
        fname = os.path.basename(file_path)
        video_id = os.path.splitext(fname)[0]
        print(f"\n[{idx}/{total}] 🎵 Đang xử lý: {fname} (ID: {video_id})")

        track_stem_dir = os.path.join(OUTPUT_DIR, video_id)
        os.makedirs(track_stem_dir, exist_ok=True)

        temp_demucs = f"/content/temp_web_{video_id}"
        os.makedirs(temp_demucs, exist_ok=True)

        try:
            cmd = [
                "python3", "-m", "demucs.separate",
                "-n", "htdemucs",
                "-d", "cuda" if torch.cuda.is_available() else "cpu",
                "-o", temp_demucs,
                file_path
            ]
            subprocess.run(cmd, check=True)

            stems_folder = os.path.join(temp_demucs, "htdemucs", os.path.splitext(fname)[0])
            if not os.path.exists(stems_folder):
                subdirs = [os.path.join(temp_demucs, "htdemucs", d) for d in os.listdir(os.path.join(temp_demucs, "htdemucs")) if os.path.isdir(os.path.join(temp_demucs, "htdemucs", d))]
                if subdirs:
                    stems_folder = subdirs[0]

            # Đo BPM & Năng lượng bằng Librosa
            print("  📊 Đang phân tích BPM và phổ âm...")
            y, sr = librosa.load(file_path, sr=22050, duration=90)
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            bpm = float(tempo[0]) if hasattr(tempo, "__iter__") else float(tempo)
            rms_energy = float(librosa.feature.rms(y=y).mean())

            print("  🎛️ Đang xuất 4 stems MP3 320kbps (vocals, other, bass, drums)...")
            other_seg = AudioSegment.from_wav(os.path.join(stems_folder, "other.wav"))
            drums_seg = AudioSegment.from_wav(os.path.join(stems_folder, "drums.wav"))
            bass_seg = AudioSegment.from_wav(os.path.join(stems_folder, "bass.wav"))
            vocals_seg = AudioSegment.from_wav(os.path.join(stems_folder, "vocals.wav"))

            min_len = min(len(other_seg), len(drums_seg), len(bass_seg), len(vocals_seg))
            vocals_seg[:min_len].export(os.path.join(track_stem_dir, "vocals.mp3"), format="mp3", bitrate="320k")
            other_seg[:min_len].export(os.path.join(track_stem_dir, "other.mp3"), format="mp3", bitrate="320k")
            bass_seg[:min_len].export(os.path.join(track_stem_dir, "bass.mp3"), format="mp3", bitrate="320k")
            drums_seg[:min_len].export(os.path.join(track_stem_dir, "drums.mp3"), format="mp3", bitrate="320k")

            track_title = fname
            try:
                oembed_url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
                req = urllib.request.Request(oembed_url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=3) as r:
                    d = json.loads(r.read().decode())
                    if d.get("title"):
                        track_title = d["title"]
            except Exception:
                pass

            meta = {
                "video_id": video_id,
                "title": track_title,
                "duration": int(min_len / 1000),
                "bpm": round(bpm, 1),
                "rms_energy": round(rms_energy, 4),
                "bitrate": "320kbps",
                "processed_by": "Google Colab HTDemucs GPU (Web ZIP)",
                "status": "READY"
            }
            with open(os.path.join(track_stem_dir, "metadata.json"), "w", encoding="utf-8") as mf:
                json.dump(meta, mf, indent=2, ensure_ascii=False)

            print(f"  ✅ Đã tách xong: {track_title} (BPM: {round(bpm, 1)})")
        except Exception as e:
            print(f"  ❌ Lỗi xử lý {fname}: {e}")
        finally:
            shutil.rmtree(temp_demucs, ignore_errors=True)

    print("\n📦 Đang nén file Stems_Completed.zip...")
    zip_out = "/content/Stems_Completed.zip"
    if os.path.exists(zip_out):
        os.remove(zip_out)

    with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, fs in os.walk(OUTPUT_DIR):
            for f in fs:
                full_p = os.path.join(root, f)
                rel_p = os.path.relpath(full_p, OUTPUT_DIR)
                z.write(full_p, rel_p)

    print(f"🎉 Hoàn tất! Kích thước file ZIP: {round(os.path.getsize(zip_out) / (1024**2), 2)} MB")
    print("⬇️ Trình duyệt sẽ tự động tải file 'Stems_Completed.zip' về máy tính ngay bây giờ...")
    files.download(zip_out)
    print("\n👉 BƯỚC TIẾP THEO: Mở ứng dụng Adaptive Music, bấm [Nạp File ZIP Stems] và kéo thả file zip vào để thưởng thức 4 Stems ngay lập tức!")